Task 1: Data Exploration and Understanding

Initialize Spark Session

In [3]:
# Importing the required libraries from pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hash
from pyspark.sql.functions import count, when

In [5]:
# Initialising spark for distributed data processing with limited memory for efficient execution
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .master("local[2]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "2") \
    .config("spark.default.parallelism", "2") \
    .getOrCreate()
print("Spark Session Created Successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/07 20:09:05 WARN Utils: Your hostname, Abhis-MacBook.local, resolves to a loopback address: 127.0.0.1; using 10.223.139.107 instead (on interface en0)
26/06/07 20:09:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 20:09:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully


Load Dataset

In [8]:
# Load transaction dataset from CSV file
df = spark.read.csv("/Users/abhinavkonda/Developer/FraudDetection/data.csv", header=True, inferSchema=True)

Inspect Dataset Schema

In [11]:
# Display schema for structure and data types 
df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



Preview Dataset

In [14]:
# Display the first 20 rows of the dataset
df.show(20)

+----+--------+---------+-------------+-------------+--------------+-------------+--------------+--------------+-------+--------------+
|step|    type|   amount|     nameOrig|oldbalanceOrg|newbalanceOrig|     nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+---------+-------------+-------------+--------------+-------------+--------------+--------------+-------+--------------+
| 371|CASH_OUT|367336.05|sdv-pii-r8zd6|   4514816.83|    2108392.86|sdv-pii-q6998|    1265486.06|    2454140.46|      0|             0|
| 368|TRANSFER|   238.63|sdv-pii-xq6z3|    430944.71|     1865444.6|sdv-pii-n2ql8|     107927.46|       2021.16|      0|             0|
| 141|CASH_OUT|   254.93|sdv-pii-805w0|    839593.53|    8008353.88|sdv-pii-yo0z6|     773352.22|         20.79|      0|             0|
| 191| CASH_IN|501547.39|sdv-pii-279tw|      41226.4|      28633.52|sdv-pii-9zlyl|    6825363.55| 1.644207824E7|      0|             0|
| 169|TRANSFER|  71832.0|sdv-pii-ksz58|     2486

Verify Dataset File Size

In [17]:
# Calculate total dataset size in MB for Big Data justification
import os

path = "/Users/abhinavkonda/Developer/CTISystem"
total_size = 0

for file in os.listdir(path):
    if file.endswith(".csv"):
        total_size += os.path.getsize(os.path.join(path, file))

print("Total size in MB:", total_size / (1024 * 1024))

# Display the total number of rows and columns in the dataset
print("Total no of Rows:", df.count())
print("Total no of Columns:", len(df.columns))

Total size in MB: 0.0


[Stage 3:================================================>        (12 + 2) / 14]

Total no of Rows: 21000000
Total no of Columns: 11


Check for missing values

In [20]:
# Check if there are any missing values in each column
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

[Stage 6:====================================================>    (13 + 1) / 14]

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



Task 2: Data Preprocessing and Feature Engineering

Handling missing values

In [23]:
# No missing values found, so no dropping of column or filling of missing value

Partition Check

In [27]:
# Check no of partitions for distributed data processing verification
print("Partitions:", df.rdd.getNumPartitions())

Partitions: 14


Sampling + Caching

In [30]:
# Sampling of dataset to reduce computation cost
df = df.sample(0.5, seed=42)
# Persist dataset in memory for faster reuse
df = df.persist()
df.count()
print("No of partitions after sampling:", df.rdd.getNumPartitions())

[Stage 9:====================================================>    (13 + 1) / 14]

No of partitions after sampling: 14


In [31]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

Type Indexing

In [35]:
# Convert transaction type from string to double 
type_indexer = StringIndexer(
    inputCol="type",
    outputCol="type_index",
    handleInvalid="keep"
)

Type Casting 

In [38]:
# Convert numeric columns to double datatype 
num_cols = ["amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]

for c in num_cols:
    df = df.withColumn(c, col(c).cast("double"))

In [66]:
df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- balance_diff_orig: double (nullable = true)
 |-- balance_diff_dest: double (nullable = true)
 |-- orig_hash: integer (nullable = false)
 |-- dest_hash: integer (nullable = false)



Feature Engineering

In [41]:
# Difference in sender balance
df = df.withColumn(
    "balance_diff_orig",
    col("oldbalanceOrg") - col("newbalanceOrig")
)
# Difference in receiver balance
df = df.withColumn(
    "balance_diff_dest",
    col("newbalanceDest") - col("oldbalanceDest")
)
# Hash encoding for account IDs
df = df.withColumn(
    "orig_hash",
    hash(col("nameOrig"))
)
df = df.withColumn(
    "dest_hash",
    hash(col("nameDest"))
)

Feature Selection

In [44]:
# List of features used for training the model
feature_cols = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "type_index",
    "balance_diff_orig",
    "balance_diff_dest",
    "orig_hash",
    "dest_hash"
]

In [62]:
df.show(5)

+----+--------+---------+-------------+-------------+--------------+-------------+--------------+--------------+-------+--------------+-----------------+------------------+-----------+-----------+
|step|    type|   amount|     nameOrig|oldbalanceOrg|newbalanceOrig|     nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|balance_diff_orig| balance_diff_dest|  orig_hash|  dest_hash|
+----+--------+---------+-------------+-------------+--------------+-------------+--------------+--------------+-------+--------------+-----------------+------------------+-----------+-----------+
| 169|TRANSFER|  71832.0|sdv-pii-ksz58|     248694.6|     793617.86|sdv-pii-0ykbo|     579313.76|     829850.96|      0|             0|       -544923.26|250537.19999999995|-1961678253| -983671157|
| 374| PAYMENT|172354.76|sdv-pii-mra6t|    176829.98|      35107.31|sdv-pii-1qflq|     299504.68|         282.5|      0|             0|        141722.67|        -299222.18| -347995910| 1039014736|
| 436| CASH_IN|

Vector Assembly

In [47]:
# Combine all the features in single vector column
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [60]:
processed_df.select("features").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                    |
+----------------------------------------------------------------------------------------------------------------------------+
|[169.0,71832.0,248694.6,793617.86,579313.76,829850.96,3.0,-544923.26,250537.19999999995,-1.961678253E9,-9.83671157E8]       |
|[374.0,172354.76,176829.98,35107.31,299504.68,282.5,1.0,141722.67,-299222.18,-3.4799591E8,1.039014736E9]                    |
|[436.0,5038.3,862919.64,1.32751028E7,4691791.26,5418837.86,2.0,-1.241218316E7,727046.6000000006,-1.048430662E9,4.96622737E8]|
|[232.0,35508.34,2071693.88,1209503.24,1437966.25,2315515.52,0.0,862190.6399999999,877549.27,1.081567419E9,-7.59276989E8]    |
|[336.0,225742.69,6439056.77,1172226.46,915394.74,1604844.94,1.0,5266830.31,689450.2,1.353037347E9,-1.72072097E

Feature Scaling

In [50]:
# Scale features so all values are in same range
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledfeatures",
    withStd=True,
    withMean=False
)

Build Pipeline

In [53]:
# Build full ML pipeline
pipeline = Pipeline(stages=[
    type_indexer,
    assembler,
    scaler
])

Train and Transform Data

In [56]:
# Train pipeline and transform the data
model = pipeline.fit(df)
processed_df = model.transform(df)

In [58]:
# Display final transformed features 
processed_df.select("scaledfeatures").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|scaledfeatures                                                                                                                                                                                                            |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[1.0353264764718486,0.2834998218215426,0.0965612815891047,0.2323916241759714,0.2309345647206019,0.08005868391735659,3.05813644819734,-0.20335383459636372,0.02856644420238684,-1.582135569719074,-0.7931765447943226]     |
|[2.291195871008706,0.6802336528301417,0.06865822374983514,0.010280319033330881,0.11939295712151418,2.72537832656760